In [ ]:
import os
import json
import chromadb
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


@dataclass
class Config:
    llm_model: str = "gpt-4o-mini"
    embed_model: str = "text-embedding-3-large"
    temperature: float = 0.0


config = Config()
client = OpenAI()   



def llm_json(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=config.llm_model,
        messages=messages,
        temperature=config.temperature,
        response_format={"type": "json_object"},
    )
    raw = resp.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start: end + 1])


def llm_text(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=config.llm_model,
        messages=messages,
        temperature=config.temperature,
    )
    return resp.choices[0].message.content.strip()


def llm_embed(text):
    try:
        resp = client.embeddings.create(
            model=config.embed_model,
            input=text,
        )
        return resp.data[0].embedding
    except Exception as e:
        print(f"[llm_embed warning] {e}")
        return None

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("OPENAI_API_KEY no encontrada. Crea el .env junto al notebook.")
else:
    print(f"OpenAI key cargada (...{api_key[-6:]})")
    print(f"  LLM   : {config.llm_model}")
    print(f"  EMBED : {config.embed_model}")
    try:
        test_emb = llm_embed("hola mundo")
        if test_emb is None:
            print("  Embeddings NO disponibles (revisa la key/billing)")
        else:
            print(f"  Embeddings OK (dim={len(test_emb)})")
        test_text = llm_text("Reply with the single word: pong")
        print(f"  LLM OK -> {test_text!r}")
    except Exception as e:
        print(f"  ERROR llamando a OpenAI: {e}")


OpenAI key cargada (...OiJfEA)
  LLM   : gpt-4o-mini
  EMBED : text-embedding-3-large
  Embeddings OK (dim=3072)
  LLM OK -> 'Pong'


# Fase 1 — Extracción de información

Por cada turno, el LLM extrae:
- **entidades nombradas** (personas, lugares, organizaciones, ...)
- **resumen factual** (preserva nombres, fechas, números)
- **tópicos** (2-5 keywords)


In [ ]:
TITLE_PREFIXES = {
    "dr", "doctor", "mr", "mister", "mrs", "ms", "miss",
    "prof", "professor", "chef", "sir", "lady", "captain", "officer",
}


def strip_title_prefix(name):
    parts = name.split("_")
    if len(parts) > 1 and parts[0] in TITLE_PREFIXES:
        return "_".join(parts[1:])
    return name


ENTITY_EXTRACTOR_PROMPT = """You are an entity extractor for a long-term memory system.

Extract ONLY named entities that have an EXPLICIT, SPECIFIC NAME in the text.

IMPORTANT: DO NOT extract "user", "assistant", "speaker", or any conversational
role as an entity.

PRONOUN RESOLUTION (NEW):
If the current turn uses a pronoun ("his", "her", "their", "its") that clearly
refers to a named entity mentioned in the PREVIOUS turn, treat the pronoun as
that entity and extract it explicitly.

Example:
  Previous turn: "Does chef Marco run his own restaurant?"
  Current turn:  "Yes, his restaurant is called La Tavola."
  -> Extract: marco (person), la_tavola (organization)
  (because "his" refers to Marco from the previous turn)

If no clear referent exists for the pronoun, do NOT extract anything for it.

For each entity, output:
- "name": canonical lowercase identifier with underscores.
- "type": one of [person, animal, location, organization, object, concept, event, date, attribute].

TYPE GUIDELINES:
- "person"       -> humans only with a specific name.
- "animal"       -> pets, named animals (dogs, cats, birds).
                    NEVER classify a named pet as "person".
- "location"     -> cities, neighborhoods, places, addresses.
- "organization" -> companies, restaurants, schools, shelters, clinics.
- "object"       -> specific products, brands.
- "concept"      -> named ideas (rare).
- "event"        -> specific named events.
- "date"         -> specific dates or days (Tuesdays, last Saturday, Thursday).
- "attribute"    -> rare; properties of someone (only if named).

CANONICAL NAMING RULES:
- ALWAYS strip titles ("Dr.", "Mr.", "Mrs.", "Prof.", "chef", ...).
  * "Dr. Maria Garcia" -> "maria_garcia"
  * "chef Marco"        -> "marco"
- Use only the proper-noun part of the name.
- Examples by type:
  * "the Eiffel Tower"   -> {{"name": "eiffel_tower", "type": "location"}}
  * "Microsoft Office"   -> {{"name": "microsoft_office", "type": "object"}}
  * "my brother John"    -> {{"name": "john", "type": "person"}}
  * "my dog Toby"        -> {{"name": "toby", "type": "animal"}}
  * "Refugio Esperanza"  -> {{"name": "refugio_esperanza", "type": "organization"}}
  * "on Tuesdays"        -> {{"name": "tuesdays", "type": "date"}}
  * "last Saturday"      -> {{"name": "last_saturday", "type": "date"}}

WHAT TO EXTRACT:
- Specific, NAMED people, animals, places, organizations, products, events, dates.
- Days of the week mentioned as recurring or specific (Tuesdays, Monday, Thursday).
- Pronouns ("his", "her", "their") that clearly refer to entities in the previous turn.

WHAT IS *NOT* AN ENTITY:
- Conversational roles ("user", "assistant", "I", "me", "you").
- Unnamed references ("my brother", "my house", "my dog" without a name).
- Generic concepts without a specific name.
- Pronouns without a clear referent in the previous turn.

If no specific named entity appears, return an empty list:
{{"entities": []}}

OUTPUT — return ONLY this JSON:
{{"entities": [{{"name": "<name>", "type": "<type>"}}, ...]}}

{previous_context_section}CURRENT TURN:
---
{conversation}
---
"""


def extract_entities(text, speaker=None, prev_context=None):
    prev_section = ""
    if prev_context:
        prev_section = f"PREVIOUS TURN (for pronoun resolution):\n{prev_context}\n\n"

    prompt = ENTITY_EXTRACTOR_PROMPT.format(
        conversation=text,
        previous_context_section=prev_section,
    )
    result = llm_json(prompt)
    cleaned = []
    for e in result.get("entities", []):
        name = e.get("name", "").strip().lower()
        if not name:
            continue
        if name in ("user", "assistant", "speaker"):
            continue
        if name.startswith(("user_", "assistant_", "speaker_")):
            continue
        name = strip_title_prefix(name)
        if not name:
            continue
        cleaned.append({"name": name, "type": e.get("type", "concept")})
    return cleaned


In [ ]:
SUMMARY_PROMPT = """You are a memory system processing a conversation turn.

Generate a 1-2 sentence summary (third-person, max ~30 words) that PRESERVES
all specific factual values: numbers, money, times, dates, names, brands,
durations, identifying details.

CRITICAL RULES:
1. SPEAKER ATTRIBUTION — the summary MUST start with "The {speaker}":
   - If speaker = "user"      -> "The user lives in..."
   - If speaker = "assistant" -> "The assistant said..."
   - NEVER say "the user" when the speaker is the assistant.

2. ONLY USE INFORMATION FROM THE CURRENT TURN — never copy facts from the
   examples below.

3. PRONOUN RESOLUTION (NEW):
   If the current turn uses pronouns ("his", "her", "their", "it") referring
   to a named entity in the PREVIOUS TURN, EXPAND the pronoun to the explicit
   name in your summary.

   Example:
     Previous turn: "Does chef Marco run his own restaurant?"
     Current turn:  "Yes, his restaurant is called La Tavola."
     GOOD summary: "The user confirmed Marco's restaurant is called La Tavola."
     BAD  summary: "The user mentioned the restaurant is called La Tavola."

4. SHORT TURNS GET SHORT SUMMARIES.

Examples (study STYLE, ignore content):

  Speaker: user
  Original: "I just got a 2020 Honda Civic for $18,500 last weekend."
  GOOD: "The user bought a 2020 Honda Civic for $18,500 last weekend."

  Speaker: user
  Original: "On Mondays I have therapy at 4pm with Dr. Lopez."
  GOOD: "The user has therapy with Dr. Lopez at 4pm on Mondays."

  Speaker: assistant
  Original: "That sounds great, congratulations on the move."
  GOOD: "The assistant congratulated the user on their move."

  Speaker: assistant
  Original: "Glovo."
  GOOD: "The assistant answered: Glovo."

Also extract 2-5 short lowercase topics relevant to the current turn.

Speaker: {speaker}

{previous_context_section}OUTPUT FORMAT — return ONLY this JSON:
{{"summary": "<factual summary starting with 'The {speaker}'>", "topics": ["<t1>", "<t2>"]}}

CURRENT TURN:
---
{turn}
---
"""


def extract_summary_topics(text, speaker, prev_context=None):
    """Extrae summary y topics. Si prev_context se pasa, el extractor
    resuelve pronombres referidos al turno anterior."""
    prev_section = ""
    if prev_context:
        prev_section = f"PREVIOUS TURN (for pronoun resolution):\n{prev_context}\n\n"

    prompt = SUMMARY_PROMPT.format(
        turn=text,
        speaker=speaker,
        previous_context_section=prev_section,
    )
    result = llm_json(prompt)
    return {
        "summary": result.get("summary", text[:80]),
        "topics": result.get("topics", []),
    }


In [ ]:
def phase1_extract(text, speaker, prev_context=None):
    entities = extract_entities(text, prev_context=prev_context)
    st = extract_summary_topics(text, speaker=speaker, prev_context=prev_context)
    return {
        "entities": entities,
        "summary": st["summary"],
        "topics": st["topics"],
    }


## Test de Fase 1



In [5]:
test_turns = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass and I go there every morning before work.",
    "I just moved from Seattle to New York for a new job at Stripe as a software engineer.",
    "Yesterday I went to a LGBTQ support group and it was really powerful.",
    "I have two kids, Ava and Noah, and they keep me busy.",
]

for i, turn in enumerate(test_turns, 1):
    print(f"\n{'='*72}\nTURNO {i}: {turn}\n{'='*72}")
    result = phase1_extract(turn, speaker="user")
    print(f"  RESUMEN  : {result['summary']}")
    print(f"  TÓPICOS  : {result['topics']}")
    print(f"  ENTIDADES ({len(result['entities'])}):")
    for e in result['entities']:
        print(f"     - {e['name']:30s} ({e['type']})")



TURNO 1: I live in San Francisco with my partner Sam.
  RESUMEN  : The user lives in San Francisco with their partner Sam.
  TÓPICOS  : ['San Francisco', 'partner']
  ENTIDADES (2):
     - san_francisco                  (location)
     - sam                            (person)

TURNO 2: My favorite coffee shop is Sightglass and I go there every morning before work.
  RESUMEN  : The user goes to their favorite coffee shop, Sightglass, every morning before work.
  TÓPICOS  : ['coffee shop', 'Sightglass', 'morning routine']
  ENTIDADES (1):
     - sightglass                     (organization)

TURNO 3: I just moved from Seattle to New York for a new job at Stripe as a software engineer.
  RESUMEN  : The user moved from Seattle to New York for a new job at Stripe as a software engineer.
  TÓPICOS  : ['moving', 'job', 'Stripe', 'software engineer']
  ENTIDADES (3):
     - seattle                        (location)
     - new_york                       (location)
     - stripe               

# Fase 2 — Actualización del grafo


In [6]:
# ADD    : el turno aporta info NUEVA -> añadir a attributes
# MODIFY : el turno CONTRADICE o ACTUALIZA info previa -> reemplazar
# KEEP   : el turno no aporta info estructural -> no tocar attributes

ENTITY_CONFLICT_PROMPT = """You are a memory system maintaining structured attributes for named entities.

Given:
  - An entity already in memory with its CURRENT attributes (may be empty)
  - A NEW turn that mentions this entity

Decide ONE action:
  - "ADD"    : new turn reveals NEW facts about this entity -> add to attributes.
  - "MODIFY" : new turn CONTRADICTS or UPDATES existing facts -> replace values.
  - "KEEP"   : no new factual attribute, just re-mention -> keep unchanged.

Attribute values must be CONCISE strings (1-5 words).

Examples:

  Entity: "toby"  | Attributes: {{}}
  Turn: "I adopted Toby last Saturday."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday"}}}}

  Entity: "toby"  | Attributes: {{"adopted_on": "last Saturday"}}
  Turn: "Toby is a two-year-old beagle from Refugio Esperanza."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday", "age": "2 years", "breed": "beagle", "origin": "Refugio Esperanza"}}}}

  Entity: "toby"  | Attributes: {{"age": "2 years", "breed": "beagle"}}
  Turn: "Actually Toby just turned 3 years old."
  -> {{"action": "MODIFY", "attributes": {{"age": "3 years", "breed": "beagle"}}}}

  Entity: "toby"  | Attributes: {{"breed": "beagle"}}
  Turn: "I took Toby to the beach today."
  -> {{"action": "KEEP", "attributes": {{"breed": "beagle"}}}}

  Entity: "glovo" | Attributes: {{}}
  Turn: "It is at a startup called Glovo and I work as a backend engineer."
  -> {{"action": "ADD", "attributes": {{"type": "startup", "user_role": "backend engineer"}}}}

ENTITY: {entity_name}
CURRENT ATTRIBUTES: {current_attributes}

NEW TURN (speaker={speaker}):
---
{turn_text}
---

Output ONLY this JSON:
{{"action": "ADD" | "MODIFY" | "KEEP", "attributes": {{...COMPLETE new attributes dict...}}}}
"""


def resolve_entity_conflict(entity_name, current_attributes, turn_text, speaker):
    try:
        prompt = ENTITY_CONFLICT_PROMPT.format(
            entity_name=entity_name,
            current_attributes=json.dumps(current_attributes),
            speaker=speaker,
            turn_text=turn_text,
        )
        result = llm_json(prompt)
        action = result.get("action", "KEEP")
        attrs = result.get("attributes", current_attributes)
        if action not in ("ADD", "MODIFY", "KEEP"):
            action = "KEEP"
            attrs = current_attributes
        if not isinstance(attrs, dict):
            attrs = current_attributes
        return {"action": action, "attributes": attrs}
    except Exception as e:
        print(f"[resolve_entity_conflict warning] {entity_name}: {e}")
        return {"action": "KEEP", "attributes": current_attributes}


In [ ]:
import math
import networkx as nx
from datetime import datetime, timezone


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def _build_embed_text(summary, topics):
    topics_str = ", ".join(topics) if topics else ""
    return f"{summary}\nTopics: {topics_str}" if topics_str else summary


class ConvMemoryGraph:

    def __init__(self,
                 alpha=0.3, beta=0.4, gamma=0.3, lam=0.05, n_max=30,
                 compute_embeddings=True,
                 enable_conflict_resolution=True):
        self.g = nx.MultiDiGraph()
        self.turn_counter = 0
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.n_max = n_max
        self.compute_embeddings = compute_embeddings
        self.enable_conflict_resolution = enable_conflict_resolution
        self._prev_statement = None   

    def add(self, text, speaker="user", kind="statement"):
        prev_context = None
        if self._prev_statement is not None:
            prev_sp, prev_txt = self._prev_statement
            prev_context = f"{prev_sp}: {prev_txt}"

        ext = phase1_extract(text, speaker, prev_context=prev_context)
        summary = ext["summary"]
        topics = ext["topics"]

        if self.compute_embeddings:
            embedding = llm_embed(_build_embed_text(summary, topics))
        else:
            embedding = None

        position = self.turn_counter
        turn_id = f"t{position}"
        self.g.add_node(
            turn_id,
            node_type="turn",
            position=position,
            summary=summary,
            topics=topics,
            role=speaker,
            kind=kind,
            is_query=(kind == "query"),
            embedding=embedding,
            r=1.0,
            created_at=now_iso(),
        )

        # Fase 2: nodos entidad + aristas MENTIONS
        n_conflict_calls = 0
        for e in ext["entities"]:
            name, etype = e["name"], e["type"]
            is_new = name not in self.g.nodes

            if is_new:
                self.g.add_node(
                    name,
                    node_type="entity",
                    entity_type=etype,
                    attributes={},
                    first_seen=position,
                    last_seen=position,
                    created_at=now_iso(),
                )
                if self.enable_conflict_resolution and kind != "query":
                    resolution = resolve_entity_conflict(
                        entity_name=name,
                        current_attributes={},
                        turn_text=text,
                        speaker=speaker,
                    )
                    n_conflict_calls += 1
                    if resolution["action"] != "KEEP" and resolution["attributes"]:
                        self.g.nodes[name]["attributes"] = resolution["attributes"]
            else:
                if self.enable_conflict_resolution and kind != "query":
                    current_attrs = self.g.nodes[name].get("attributes", {})
                    resolution = resolve_entity_conflict(
                        entity_name=name,
                        current_attributes=current_attrs,
                        turn_text=text,
                        speaker=speaker,
                    )
                    n_conflict_calls += 1
                    if resolution["action"] != "KEEP":
                        self.g.nodes[name]["attributes"] = resolution["attributes"]
                self.g.nodes[name]["last_seen"] = position

            self.g.add_edge(turn_id, name,
                            edge_type="MENTIONS",
                            created_at=now_iso())

        # Recalcular r(t_i) 
        t_actual = position
        for tid in self._turn_ids():
            self.g.nodes[tid]["r"] = self._compute_r(tid, t_actual)

        # Poda
        n_pruned = 0
        n_orphans = 0
        if self._n_turns() > self.n_max:
            n_pruned = self._prune_low_relevance_turns()
            n_orphans = self._prune_orphan_entities()

        if kind == "statement":
            self._prev_statement = (speaker, text)

        self.turn_counter += 1
        return {
            "turn_id": turn_id,
            "position": position,
            "role": speaker,
            "kind": kind,
            "n_entities": len(ext["entities"]),
            "n_conflict_calls": n_conflict_calls,
            "has_embedding": embedding is not None,
            "n_pruned": n_pruned,
            "n_orphans_removed": n_orphans,
        }

    # r(t_i) = α·ant + β·men + γ·ult
    def _compute_r(self, turn_id, t_actual):
        i = self.g.nodes[turn_id]["position"]

        # Antigüedad
        ant = math.exp(-self.lam * (t_actual - i))

        # Menciones: cuántos turnos del grafo comparten alguna entidad con este.
        entities_i = self._entities_of_turn(turn_id)
        sharing_count = 0
        last_mention_j = i

        for tid in self._turn_ids():
            j = self.g.nodes[tid]["position"]
            if j < i:
                continue
            if j == i:
                sharing_count += 1
                continue
            entities_j = self._entities_of_turn(tid)
            if entities_i & entities_j:
                sharing_count += 1
                last_mention_j = max(last_mention_j, j)

        men = sharing_count / max(1, t_actual + 1)

        # Última aparición
        ult = math.exp(-self.lam * (t_actual - last_mention_j))

        return self.alpha * ant + self.beta * men + self.gamma * ult

    def _turn_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "turn"]

    def _entity_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "entity"]

    def _n_turns(self):
        return len(self._turn_ids())

    def _entities_of_turn(self, turn_id, exclude_speaker=None):
        out = set()
        for _, v, d in self.g.out_edges(turn_id, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            out.add(v)
        return out

    def _turns_mentioning_entity(self, entity_name, include_queries=False):
        if entity_name not in self.g.nodes:
            return []
        turns = []
        for u, _, d in self.g.in_edges(entity_name, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            node = self.g.nodes[u]
            if node.get("node_type") != "turn":
                continue
            if not include_queries and node.get("is_query"):
                continue
            turns.append(u)
        return turns

    def _prune_low_relevance_turns(self):
        n_to_remove = self._n_turns() - self.n_max
        if n_to_remove <= 0:
            return 0
        turns_sorted = sorted(self._turn_ids(),
                              key=lambda tid: self.g.nodes[tid]["r"])
        for tid in turns_sorted[:n_to_remove]:
            self.g.remove_node(tid)
        return n_to_remove

    def _prune_orphan_entities(self):
        to_remove = []
        for eid in self._entity_ids():
            if self.g.in_degree(eid) > 0:
                continue
            if self.g.nodes[eid].get("attributes"):
                continue
            to_remove.append(eid)
        for eid in to_remove:
            self.g.remove_node(eid)
        return len(to_remove)

    def show_state(self):
        print(f"\n{'='*72}")
        print(f"ESTADO DEL GRAFO  |  turnos: {self._n_turns()}  "
              f"|  entidades: {len(self._entity_ids())}")
        print('='*72)

        print(f"\nNODOS DE TURNO:")
        for tid in sorted(self._turn_ids(),
                          key=lambda x: self.g.nodes[x]["position"]):
            d = self.g.nodes[tid]
            menciona = sorted(self._entities_of_turn(tid))
            bar = "█" * int(d["r"] * 20) + "·" * (20 - int(d["r"] * 20))
            kind_tag = " [QUERY]" if d.get("is_query") else ""
            emb_tag = "·emb" if d.get("embedding") else ""
            print(f"  [{tid}] pos={d['position']}  role={d.get('role','?')}{kind_tag}{emb_tag}  "
                  f"r={d['r']:.3f}  {bar}")
            print(f"        resumen:  {d['summary']}")
            print(f"        tópicos:  {d['topics']}")
            print(f"        menciona: {menciona}")
            print()

        print(f"NODOS DE ENTIDAD:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentioning = sorted(self._turns_mentioning_entity(eid,
                                                              include_queries=True))
            attrs = d.get("attributes", {})
            print(f"  [{eid}]  tipo={d['entity_type']}  "
                  f"first=t{d['first_seen']}, last=t{d['last_seen']}  "
                  f"mencionada por: {mentioning}")
            if attrs:
                print(f"        attributes: {attrs}")

    def show_state_compact(self, last_n=None):
        n_t = self._n_turns()
        n_e = len(self._entity_ids())
        print(f"\n{'─'*72}")
        print(f"GRAFO  |  turnos: {n_t}  |  entidades: {n_e}")
        print('─'*72)

        tids = sorted(self._turn_ids(),
                      key=lambda x: self.g.nodes[x]["position"])
        if last_n is not None and len(tids) > last_n:
            print(f"  ... ({len(tids) - last_n} turnos anteriores omitidos)")
            tids = tids[-last_n:]

        for tid in tids:
            d = self.g.nodes[tid]
            tag = "Q" if d.get("is_query") else "S"
            print(f"  [{tid:>4}] pos={d['position']:>2} {tag} "
                  f"role={d.get('role','?'):<9} r={d['r']:.2f}  "
                  f"{d['summary'][:55]}")

        print(f"\n  Entidades:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentions = self._turns_mentioning_entity(eid, include_queries=True)
            attrs = d.get("attributes", {})
            attr_str = ""
            if attrs:
                attr_str = "  " + ", ".join(f"{k}={v}" for k, v in attrs.items())
            print(f"     {eid:<30s} ({d['entity_type']:<12}) "
                  f"-> {len(mentions)} turnos: {sorted(mentions)}{attr_str}")


# Fase 3 — Recuperación de contexto + Respuesta del LLM


In [ ]:
ANSWER_PROMPT_C1 = """You are a memory assistant that knows everything about a specific user.

The MEMORIES below are summaries of past turns of conversation with the user,
sorted in REVERSE CHRONOLOGICAL ORDER.

Each turn may include a "Mentions:" section listing named entities. When an
entity has attributes shown as key=value pairs, those are STRUCTURED FACTS the
system has accumulated about that entity across the whole conversation history.
Use those facts when answering questions about that entity.

ANSWER STYLE — IMPORTANT:
- Address the user directly in second person ("you", "your").
- Answer with a complete, natural sentence (not just one word).
  GOOD: "You usually run 8 kilometers."
  BAD : "8 kilometers."
- Keep the answer short and friendly (1-2 sentences max).
- DO NOT add disclaimers like "based on the memories..." or "according to..."
- Speak as if you remember everything yourself.

ANSWER RULES:
1. Match the answer to the question's intent:
   - "Where do you live?" -> a geographic place.
   - "Where do you work?" -> an EMPLOYER (company), not a city.
   - "Where did you buy ...?" -> a store or location of purchase.
   - "Who ..." -> a person's name.
   - "How many / how much ..." -> a number or amount.
   - "When ..." -> a date or time.
   - "What sport / hobby / activity ...?" -> the SPECIFIC activity name.
   - "Why ..." -> a CAUSE that may be in a different turn than the question's
     entity. Look across all memories for the cause.
2. CONNECT MEMORIES: the answer may require combining facts from different
   turns and entity attributes.
3. RECENCY: if memories conflict, trust the more recent (higher position).
4. NEVER output an internal identifier in snake_case (e.g. "user_home").
   Rephrase in natural language.
5. Only say "I don't know" if the memories truly contain no relevant info.

MEMORIES (MOST RECENT FIRST):
{context}

QUESTION: {query}

ANSWER:"""


def cosine_sim(a, b):
    if not a or not b:
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


def build_context(mem, turn_ids):
    sorted_tids = sorted(turn_ids,
                         key=lambda t: mem.g.nodes[t]["position"],
                         reverse=True)
    lines = []
    for tid in sorted_tids:
        d = mem.g.nodes[tid]
        lines.append(f"[Turn {d['position']} | role={d.get('role','?')} | r={d['r']:.2f}]")
        lines.append(f"Summary: {d['summary']}")
        mentions = sorted(mem._entities_of_turn(tid))
        if mentions:
            lines.append("Mentions:")
            for m in mentions:
                ent_node = mem.g.nodes[m]
                etype = ent_node.get("entity_type", "?")
                attrs = ent_node.get("attributes", {}) or {}
                if attrs:
                    attr_str = ", ".join(f"{k}={v}" for k, v in attrs.items())
                    lines.append(f"  - {m} ({etype}): {attr_str}")
                else:
                    lines.append(f"  - {m} ({etype})")
        lines.append("")
    return "\n".join(lines).strip()


def retrieve_relevant_turns(mem, query, speaker="user",
                            k=5, w_sim=0.7, w_r=0.3):
  
    # 1) Embedding de la query
    ext = phase1_extract(query, speaker)
    summary_q = ext["summary"]
    topics_q = ext["topics"]
    q_emb = llm_embed(_build_embed_text(summary_q, topics_q))

   
    candidates = [
        tid for tid in mem._turn_ids()
        if not mem.g.nodes[tid].get("is_query")
    ]

    # Score híbrido 
    breakdown = {}
    for tid in candidates:
        node = mem.g.nodes[tid]
        sim = max(0.0, cosine_sim(q_emb, node.get("embedding"))) if q_emb else 0.0
        r_val = node.get("r", 0.0)
        score = w_sim * sim + w_r * r_val
        breakdown[tid] = (score, sim, r_val)

    # 4) Top-k por score
    ranked = sorted(candidates, key=lambda t: breakdown[t][0], reverse=True)
    top_k = ranked[:k]

    q_info = {
        "summary": summary_q,
        "topics": topics_q,
        "entities": sorted(set(e["name"] for e in ext["entities"])),
    }
    return q_info, top_k, breakdown


def answer(mem, query, speaker="user",
           k=5, w_sim=0.7, w_r=0.3,
           verbose=False, return_meta=False):
    q_info, top_turns, breakdown = retrieve_relevant_turns(
        mem, query, speaker=speaker, k=k, w_sim=w_sim, w_r=w_r,
    )
    if not top_turns:
        text = "I don't know — no memories available yet."
    else:
        context = build_context(mem, top_turns)
        prompt = ANSWER_PROMPT_C1.format(context=context, query=query)
        text = llm_text(prompt)

    if verbose:
        print("=== INFO DE LA PREGUNTA (no añadida al grafo) ===")
        print(f"  resumen: {q_info['summary']}")
        print(f"  entidades: {q_info['entities']}")
        print(f"  pesos: w_sim={w_sim}, w_r={w_r}, k={k}  (sin MMR, sin entity_match)")

        print(f"\n=== TOP-{len(top_turns)} (score híbrido puro) ===")
        for tid in top_turns:
            score, sim, r_val = breakdown[tid]
            summary = mem.g.nodes[tid]["summary"]
            ents = sorted(mem._entities_of_turn(tid))
            ents_str = ",".join(ents) if ents else "—"
            print(f"  {tid}  score={score:.3f}  (sim={sim:.3f}, r={r_val:.3f})  "
                  f"[{ents_str[:30]}]")
            print(f"        {summary[:65]}")

        print("\n=== CONTEXTO ENVIADO AL LLM ===")
        print(build_context(mem, top_turns) if top_turns else "(vacío)")
        print(f"\nQUESTION: {query}")
        print("=" * 50)

    if return_meta:
        return {"text": text, "top_turn_ids": list(top_turns), "breakdown": breakdown}
    return text


In [ ]:
JUDGE_PROMPT = """You are an impartial judge for a memory QA system.

Given a QUESTION, the GOLD ANSWER, and a PREDICTED ANSWER, decide if the
predicted answer is CORRECT.

Be LENIENT with formatting: the prediction may be a full sentence while the
gold is a short fact. What matters is whether the prediction conveys the
SAME FACTUAL ANSWER as the gold.

Be STRICT with content: a vague or "I don't know" answer is wrong, even if
the topic is right.

Examples:

  GOLD: "GPS system not functioning correctly"
  PRED: "The first issue was a problem with the GPS system."
  -> YES (same fact, just rephrased)

  GOLD: "8 kilometers"
  PRED: "About 8 km"
  -> YES (same number)

  GOLD: "Glovo"
  PRED: "He works at Glovo, a delivery startup."
  -> YES (mentions Glovo)

  GOLD: "Saturday"
  PRED: "Last weekend"
  -> NO (less specific)

  GOLD: "Refugio Esperanza"
  PRED: "An animal shelter"
  -> NO (correct concept but not the name)

  GOLD: "knee injury"
  PRED: "I don't know."
  -> NO (no answer given)

  GOLD: "Yes"
  PRED: "Based on the memories, yes."
  -> YES (same verdict)

QUESTION: {question}
GOLD: {gold}
PRED: {pred}

Reply with ONLY this JSON:
{{"verdict": "YES" | "NO", "reason": "<one short sentence>"}}
"""


def judge_answer(question, gold, pred):
    """Devuelve {correct: bool, verdict: 'YES'/'NO', reason: str}."""
    if not pred or not str(pred).strip():
        return {"correct": False, "verdict": "NO", "reason": "Empty prediction"}
    try:
        result = llm_json(JUDGE_PROMPT.format(
            question=question,
            gold=gold,
            pred=pred,
        ))
        verdict = str(result.get("verdict", "NO")).upper().strip()
        return {
            "correct": verdict == "YES",
            "verdict": verdict,
            "reason": result.get("reason", "")[:200],
        }
    except Exception as e:
        return {"correct": False, "verdict": "ERROR", "reason": str(e)[:200]}


In [10]:
CONVERSATION_PATH = "test_conversation.txt"


def load_conversation(path):
    turns = []
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("USER:"):
                turns.append(("user", line[len("USER:"):].strip()))
            elif line.startswith("ASSISTANT:"):
                turns.append(("assistant", line[len("ASSISTANT:"):].strip()))
    return turns



turns = load_conversation(CONVERSATION_PATH)
print(f"Dataset cargado: {len(turns)} turnos\n")

# Crear grafo vacío con lam=0.05 (decay suave)
mem_txt = ConvMemoryGraph(alpha=0.3, beta=0.4, gamma=0.3, lam=0.05, n_max=50)

for i, (role, text) in enumerate(turns):
    print(f"{'─'*72}")
    print(f"TURNO {i}  [role={role}]  {text}")

    ext = phase1_extract(text, speaker=role)
    ents = ", ".join(f"{e['name']}({e['type']})" for e in ext["entities"])
    print(f"  RESUMEN  : {ext['summary']}")
    print(f"  TÓPICOS  : {ext['topics']}")
    print(f"  ENTIDADES: {ents}")

    r = mem_txt.add(text, speaker=role)
    print(f"  -> {r['turn_id']} (pos={r['position']}, {r['n_entities']} entidades)\n")

print("\n" + "#"*72)
print("# GRAFO RESULTANTE")
print("#"*72)
mem_txt.show_state()


Dataset cargado: 26 turnos

────────────────────────────────────────────────────────────────────────
TURNO 0  [role=user]  I just moved from Valencia to Barcelona last week for a new job.
  RESUMEN  : The user moved from Valencia to Barcelona last week for a new job.
  TÓPICOS  : ['moving', 'job', 'Valencia', 'Barcelona']
  ENTIDADES: valencia(location), barcelona(location)
  -> t0 (pos=0, 2 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 1  [role=assistant]  Congratulations on the move to Barcelona. How is the new job going so far?
  RESUMEN  : The assistant congratulated the user on their move to Barcelona and inquired about the new job's progress.
  TÓPICOS  : ['Barcelona', 'new job', 'congratulations']
  ENTIDADES: barcelona(location)
  -> t1 (pos=1, 1 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 2  [role=user]  It is at a startup called Glovo and I work as a backend engineer.
  RESUMEN  : The 

In [ ]:
preguntas_gold = [
    ("What sport do I do regularly?",                                "running"),
    ("What day do I have cooking class?",                            "Tuesdays"),
    ("What is my role at work?",                                     "backend engineer"),
    ("How many kilometers do I usually run?",                        "8 kilometers"),
    ("Where do I work?",                                             "Glovo"),
    ("Who is my manager at Glovo?",                                  "Elena"),
    ("Why do I need to see Dr. Ferrer?",                             "knee injury"),
    ("In what neighborhood is La Tavola?",                           "Gracia"),
    ("What's the connection between Marco and La Tavola?",           "Marco owns La Tavola"),
    ("In what city is Clinica Diagonal?",                            "Barcelona"),
    ("What is Toby's age?",                                          "2 years old"),
    ("What breed is Toby?",                                          "beagle"),
    ("When did I adopt my dog?",                                     "last Saturday"),
    ("Where does my dog Toby come from?",                            "Refugio Esperanza"),
]

n_turns_antes = mem_txt._n_turns()
print(f"Turnos en el grafo antes del chat: {n_turns_antes}\n")
print("MODO EVAL READ-ONLY: ni preguntas ni respuestas se añaden al grafo.\n")

resultados_chat = []

for i, (q, gold) in enumerate(preguntas_gold, 1):
    print("\n" + "█"*72)
    print(f"# PREGUNTA {i}/{len(preguntas_gold)}")
    print("█"*72)

    ans = answer(mem_txt, q, k=5, verbose=True)
    print(f"\n>>> RESPUESTA DEL LLM: {ans}")

    # Verificación automática con el juez
    jud = judge_answer(q, gold, ans)
    estado = "[CORRECTO]" if jud["correct"] else "[INCORRECTO]"
    print(f"\n>>> GOLD ESPERADO  : {gold}")
    print(f">>> VEREDICTO JUEZ : {jud['verdict']}  {estado}")
    print(f">>> RAZÓN          : {jud['reason']}")

    resultados_chat.append({
        "i": i,
        "question": q,
        "gold": gold,
        "pred": ans,
        "correct": jud["correct"],
        "verdict": jud["verdict"],
        "reason": jud["reason"],
    })



Turnos en el grafo antes del chat: 26

MODO EVAL READ-ONLY: ni preguntas ni respuestas se añaden al grafo.


████████████████████████████████████████████████████████████████████████
# PREGUNTA 1/14
████████████████████████████████████████████████████████████████████████
=== INFO DE LA PREGUNTA (no añadida al grafo) ===
  resumen: The user asked about the sport they do regularly.
  entidades: []
  pesos: w_sim=0.7, w_r=0.3, k=5  (sin MMR, sin entity_match)

=== TOP-5 (score híbrido puro) ===
  t12  score=0.420  (sim=0.450, r=0.352)  [parc_de_la_ciutadella]
        The user has been running every morning in Parc de la Ciutadella.
  t13  score=0.399  (sim=0.423, r=0.345)  [parc_de_la_ciutadella]
        The assistant commented that Parc de la Ciutadella is a beautiful
  t24  score=0.397  (sim=0.303, r=0.616)  [barceloneta,toby]
        The user will take Toby to the beach in Barceloneta this weekend.
  t19  score=0.395  (sim=0.295, r=0.630)  [toby]
        The assistant asked what breed T

In [12]:
# ----------------------------------------------------------------------
# Resumen final con métrica de accuracy en MODO EVAL READ-ONLY
# El grafo NO debe haber crecido durante el chat (intacto).
# ----------------------------------------------------------------------
n_turns_despues = mem_txt._n_turns()
n_correctas = sum(1 for r in resultados_chat if r["correct"])
n_total = len(resultados_chat)
accuracy = n_correctas / n_total * 100 if n_total else 0

print("\n\n" + "="*72)
print("CHAT TERMINADO  (modo eval read-only)")
print("="*72)
print(f"  Turnos antes  : {n_turns_antes}")
print(f"  Turnos después: {n_turns_despues}  (delta esperado = 0)")
if n_turns_despues != n_turns_antes:
    print(f"  ATENCION: el grafo creció. Algo se está añadiendo durante el eval.")
else:
    print(f"  OK: el grafo permaneció intacto durante el eval.")

print(f"\n{'='*72}")
print(f"ACCURACY SOBRE LAS {n_total} PREGUNTAS")
print(f"{'='*72}")
print(f"  Correctas : {n_correctas}/{n_total}  =  {accuracy:.1f}%")
print(f"  Fallidas  : {n_total - n_correctas}/{n_total}")

print(f"\nDETALLE POR PREGUNTA:")
print(f"  {'#':>3}  {'estado':<12}  {'pregunta':<55}")
print(f"  {'-'*3}  {'-'*12}  {'-'*55}")
for r in resultados_chat:
    estado = "[CORRECTO]" if r["correct"] else "[INCORRECTO]"
    pregunta = r["question"][:53] + ".." if len(r["question"]) > 55 else r["question"]
    print(f"  {r['i']:>3}  {estado:<12}  {pregunta:<55}")

fallos = [r for r in resultados_chat if not r["correct"]]
if fallos:
    print(f"\n--- DETALLE DE LOS {len(fallos)} FALLOS ---")
    for r in fallos:
        print(f"\n[{r['i']}] {r['question']}")
        print(f"  GOLD: {r['gold']}")
        print(f"  PRED: {r['pred']}")
        print(f"  juez: {r['verdict']} — {r['reason']}")
else:
    print("\nTodas las preguntas correctas.")

print("\n### ESTADO FINAL DEL GRAFO (vista compacta, últimos 40 turnos)")
mem_txt.show_state_compact(last_n=40)




CHAT TERMINADO  (modo eval read-only)
  Turnos antes  : 26
  Turnos después: 26  (delta esperado = 0)
  OK: el grafo permaneció intacto durante el eval.

ACCURACY SOBRE LAS 14 PREGUNTAS
  Correctas : 14/14  =  100.0%
  Fallidas  : 0/14

DETALLE POR PREGUNTA:
    #  estado        pregunta                                               
  ---  ------------  -------------------------------------------------------
    1  [CORRECTO]    What sport do I do regularly?                          
    2  [CORRECTO]    What day do I have cooking class?                      
    3  [CORRECTO]    What is my role at work?                               
    4  [CORRECTO]    How many kilometers do I usually run?                  
    5  [CORRECTO]    Where do I work?                                       
    6  [CORRECTO]    Who is my manager at Glovo?                            
    7  [CORRECTO]    Why do I need to see Dr. Ferrer?                       
    8  [CORRECTO]    In what neighborhood is L

In [13]:
CONVERSATION_PATH = "test_conversation_2.txt"


def load_conversation(path):

    turns = []
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("USER:"):
                turns.append(("user", line[len("USER:"):].strip()))
            elif line.startswith("ASSISTANT:"):
                turns.append(("assistant", line[len("ASSISTANT:"):].strip()))
    return turns



turns = load_conversation(CONVERSATION_PATH)
print(f"Dataset cargado: {len(turns)} turnos\n")

# Crear grafo vacío
mem_txt = ConvMemoryGraph(alpha=0.2, beta=0.5, gamma=0.3, lam=0.15, n_max=50) 
for i, (role, text) in enumerate(turns):
    print(f"{'─'*72}")
    print(f"TURNO {i}  [role={role}]  {text}")

    ext = phase1_extract(text, speaker=role)
    ents = ", ".join(f"{e['name']}({e['type']})" for e in ext["entities"])
    print(f"  RESUMEN  : {ext['summary']}")
    print(f"  TÓPICOS  : {ext['topics']}")
    print(f"  ENTIDADES: {ents}")

    r = mem_txt.add(text, speaker=role)
    print(f"  -> {r['turn_id']} (pos={r['position']}, {r['n_entities']} entidades)\n")

print("\n" + "#"*72)
print("# GRAFO RESULTANTE")
print("#"*72)
mem_txt.show_state()

Dataset cargado: 30 turnos

────────────────────────────────────────────────────────────────────────
TURNO 0  [role=user]  I just moved from Madrid to Lisboa three weeks ago for a new role.
  RESUMEN  : The user moved from Madrid to Lisboa three weeks ago for a new role.
  TÓPICOS  : ['moving', 'Madrid', 'Lisboa', 'new role']
  ENTIDADES: madrid(location), lisboa(location)
  -> t0 (pos=0, 2 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 1  [role=assistant]  Welcome to Lisboa. What kind of role are you starting?
  RESUMEN  : The assistant welcomed the user to Lisboa and inquired about the kind of role the user is starting.
  TÓPICOS  : ['Lisboa', 'role', 'welcome']
  ENTIDADES: lisboa(location)
  -> t1 (pos=1, 1 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 2  [role=user]  I joined a fintech called Revolut as a data engineer.
  RESUMEN  : The user joined a fintech company called Revolut as a data en

In [14]:
preguntas_gold = [
    ("What instrument am I learning?",                          "saxophone"),
    ("What day are my music lessons?",                          "Wednesday"),
    ("What is my role at work?",                                "data engineer"),
    ("Where do I work?",                                        "Revolut"),
    ("Who is my team lead?",                                    "Marta"),
    ("Where do I live now?",                                    "Lisboa"),
    ("Why did I go to Dra. Ribeiro?",                           "broken arm"),
    ("Where does maestro Joaquim perform?",                     "Hot Clube de Portugal"),
    ("What's the connection between Joaquim and Hot Clube?",    "Joaquim performs there"),
    ("In what city is Hospital da Luz?",                        "Lisboa"),
    ("How much does Mateus weigh?",                             "3.4 kilograms"),
    ("Who is Carolina?",                                        "my wife"),
    ("When was Mateus born?",                                   "last Friday"),
    ("Where was Mateus born?",                                  "Hospital da Luz"),
    ("I have a cat, what's its name?",                          "Mia"),
]

n_turns_antes = mem_txt._n_turns()
print(f"Turnos en el grafo antes del chat: {n_turns_antes}\n")
print("MODO EVAL READ-ONLY: ni preguntas ni respuestas se añaden al grafo.\n")

resultados_chat = []

for i, (q, gold) in enumerate(preguntas_gold, 1):
    print("\n" + "█"*72)
    print(f"# PREGUNTA {i}/{len(preguntas_gold)}")
    print("█"*72)

    ans = answer(mem_txt, q, k=5, verbose=True)
    print(f"\n>>> RESPUESTA DEL LLM: {ans}")

    # Verificación automática con el juez
    jud = judge_answer(q, gold, ans)
    estado = "[CORRECTO]" if jud["correct"] else "[INCORRECTO]"
    print(f"\n>>> GOLD ESPERADO  : {gold}")
    print(f">>> VEREDICTO JUEZ : {jud['verdict']}  {estado}")
    print(f">>> RAZÓN          : {jud['reason']}")

    resultados_chat.append({
        "i": i,
        "question": q,
        "gold": gold,
        "pred": ans,
        "correct": jud["correct"],
        "verdict": jud["verdict"],
        "reason": jud["reason"],
    })


Turnos en el grafo antes del chat: 30

MODO EVAL READ-ONLY: ni preguntas ni respuestas se añaden al grafo.


████████████████████████████████████████████████████████████████████████
# PREGUNTA 1/15
████████████████████████████████████████████████████████████████████████
=== INFO DE LA PREGUNTA (no añadida al grafo) ===
  resumen: The user is learning an instrument.
  entidades: []
  pesos: w_sim=0.7, w_r=0.3, k=5  (sin MMR, sin entity_match)

=== TOP-5 (score híbrido puro) ===
  t8  score=0.594  (sim=0.680, r=0.392)  [joaquim]
        The user is learning jazz standards from maestro Joaquim.
  t7  score=0.474  (sim=0.662, r=0.035)  [—]
        The assistant commented on the beauty of the saxophone and asked 
  t6  score=0.460  (sim=0.643, r=0.033)  [wednesday]
        The user started saxophone lessons on Wednesday evenings.
  t28  score=0.414  (sim=0.375, r=0.505)  [carolina,hot_clube,joaquim]
        The user wants to take Carolina to Hot Clube to see Joaquim perfo
  t9  score=0.398 

In [15]:
# ----------------------------------------------------------------------
# Resumen final con métrica de accuracy en MODO EVAL READ-ONLY
# El grafo NO debe haber crecido durante el chat (intacto).
# ----------------------------------------------------------------------
n_turns_despues = mem_txt._n_turns()
n_correctas = sum(1 for r in resultados_chat if r["correct"])
n_total = len(resultados_chat)
accuracy = n_correctas / n_total * 100 if n_total else 0

print("\n\n" + "="*72)
print("CHAT TERMINADO  (modo eval read-only)")
print("="*72)
print(f"  Turnos antes  : {n_turns_antes}")
print(f"  Turnos después: {n_turns_despues}  (delta esperado = 0)")
if n_turns_despues != n_turns_antes:
    print(f"  ATENCION: el grafo creció. Algo se está añadiendo durante el eval.")
else:
    print(f"  OK: el grafo permaneció intacto durante el eval.")

print(f"\n{'='*72}")
print(f"ACCURACY SOBRE LAS {n_total} PREGUNTAS")
print(f"{'='*72}")
print(f"  Correctas : {n_correctas}/{n_total}  =  {accuracy:.1f}%")
print(f"  Fallidas  : {n_total - n_correctas}/{n_total}")

print(f"\nDETALLE POR PREGUNTA:")
print(f"  {'#':>3}  {'estado':<12}  {'pregunta':<55}")
print(f"  {'-'*3}  {'-'*12}  {'-'*55}")
for r in resultados_chat:
    estado = "[CORRECTO]" if r["correct"] else "[INCORRECTO]"
    pregunta = r["question"][:53] + ".." if len(r["question"]) > 55 else r["question"]
    print(f"  {r['i']:>3}  {estado:<12}  {pregunta:<55}")

fallos = [r for r in resultados_chat if not r["correct"]]
if fallos:
    print(f"\n--- DETALLE DE LOS {len(fallos)} FALLOS ---")
    for r in fallos:
        print(f"\n[{r['i']}] {r['question']}")
        print(f"  GOLD: {r['gold']}")
        print(f"  PRED: {r['pred']}")
        print(f"  juez: {r['verdict']} — {r['reason']}")
else:
    print("\nTodas las preguntas correctas.")

print("\n### ESTADO FINAL DEL GRAFO (vista compacta, últimos 40 turnos)")
mem_txt.show_state_compact(last_n=40)




CHAT TERMINADO  (modo eval read-only)
  Turnos antes  : 30
  Turnos después: 30  (delta esperado = 0)
  OK: el grafo permaneció intacto durante el eval.

ACCURACY SOBRE LAS 15 PREGUNTAS
  Correctas : 13/15  =  86.7%
  Fallidas  : 2/15

DETALLE POR PREGUNTA:
    #  estado        pregunta                                               
  ---  ------------  -------------------------------------------------------
    1  [CORRECTO]    What instrument am I learning?                         
    2  [CORRECTO]    What day are my music lessons?                         
    3  [INCORRECTO]  What is my role at work?                               
    4  [INCORRECTO]  Where do I work?                                       
    5  [CORRECTO]    Who is my team lead?                                   
    6  [CORRECTO]    Where do I live now?                                   
    7  [CORRECTO]    Why did I go to Dra. Ribeiro?                          
    8  [CORRECTO]    Where does maestro Joaquim